In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import cv2
import os
from collections import defaultdict

IMG_WIDTH, IMG_HEIGHT = 1920, 1080
VIDEO_FPS = 60000 / 1001
VIDEO_DURATION_S = 422.72
VIDEO_PATH = "/kaggle/input/datasets/natair/chicken-drone-video/drone_video.mp4"

TELEMETRY_CSV = "/kaggle/input/datasets/natair/djiflightrecord-2026-04-02-17-00-04-1/DJIFlightRecord_2026-04-02_17-00-04.csv"
VIDEO_OFFSET_S = 3.0

FOCAL_PX = 1500
NEAR_CENTER_RADIUS_PX = 150

FRAME_STRIDE = 10
MIN_INLIERS = 30
MIN_RATIO = 0.4
MAX_LOOKBACK = 6

In [ ]:
def latlng_to_local_m(lat, lng, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    return (lng - ref_lng) * m_per_deg_lng, (lat - ref_lat) * m_per_deg_lat

def local_m_to_latlng(east, north, ref_lat, ref_lng):
    m_per_deg_lat = 111320.0
    m_per_deg_lng = 111320.0 * np.cos(np.radians(ref_lat))
    return ref_lat + north / m_per_deg_lat, ref_lng + east / m_per_deg_lng

telemetry = pd.read_csv(TELEMETRY_CSV)[["time_s", "lat", "lng", "height_m", "yaw_deg"]].dropna().reset_index(drop=True)
REF_LAT, REF_LNG = telemetry["lat"].mean(), telemetry["lng"].mean()

def get_drone_state(frame):
    t = frame / VIDEO_FPS + VIDEO_OFFSET_S
    idx = (telemetry["time_s"] - t).abs().idxmin()
    row = telemetry.loc[idx]
    return row["lat"], row["lng"], row["height_m"], row["yaw_deg"]


In [ ]:
# Detections from tracks_dense.xml
TRACKS_XML = "/kaggle/input/datasets/natair/chicken-trainings-data/tracks_dense.xml"

if not os.path.exists(TRACKS_XML):
    print(f"{TRACKS_XML} not found")
    df = pd.DataFrame()
else:
    tree = ET.parse(TRACKS_XML)
    root = tree.getroot()
    rows = []
    for track in root.findall(".//track"):
        track_id = track.get("id")
        for box in track.findall("box"):
            if box.get("outside") == "1":
                continue
            frame = int(box.get("frame"))
            xtl, ytl, xbr, ybr = (float(box.get(k)) for k in ("xtl", "ytl", "xbr", "ybr"))
            cx, cy = (xtl + xbr) / 2, (ytl + ybr) / 2
            size_px = np.sqrt((xbr - xtl) * (ybr - ytl))
            dx_px, dy_px = cx - IMG_WIDTH / 2, cy - IMG_HEIGHT / 2
            r_px = np.hypot(dx_px, dy_px)
            lat, lng, h, yaw = get_drone_state(frame)
            rows.append({
                "track_id": track_id, "frame": frame, "time_s": frame / VIDEO_FPS,
                "cx": cx, "cy": cy, "size_px": size_px, "dx_px": dx_px, "dy_px": dy_px, "r_px": r_px,
                "lat": lat, "lng": lng, "height_m": h, "yaw_deg": yaw,
            })
    df = pd.DataFrame(rows)
    print(f"Total detections: {len(df)} | unique tracks: {df['track_id'].nunique() if len(df) else 0}")


Total detections:220797 | unique tracks: 278

In [ ]:
if len(df):
    near_center = df[df["r_px"] < NEAR_CENTER_RADIUS_PX]
    K_samples = near_center["size_px"] * near_center["height_m"]
    K = K_samples.median()
    spread_pct = K_samples.std() / K * 100 if K > 0 else float("nan")
    print(f"K = {K:.1f}  (median from {len(K_samples)}, range {spread_pct:.0f}%)")

    df["R"] = K / df["size_px"]
    df["valid"] = df["R"] > df["height_m"]
    df = df[df["valid"]].copy()s
    df["D"] = np.sqrt(df["R"]**2 - df["height_m"]**2)
    df["bearing_deg"] = df["yaw_deg"] + np.degrees(np.arctan2(df["dx_px"], -df["dy_px"]))

    drone_east, drone_north = latlng_to_local_m(df["lat"].values, df["lng"].values, REF_LAT, REF_LNG)
    df["drone_east"], df["drone_north"] = drone_east, drone_north
    df["pytha_east"] = drone_east + df["D"] * np.sin(np.radians(df["bearing_deg"]))
    df["pytha_north"] = drone_north + df["D"] * np.cos(np.radians(df["bearing_deg"]))
    print(f"Pythagoras is calculated for {len(df)} detections")

Pythagoras is calculated for 97809 detections

In [ ]:
_frame_cache = {}
_cap = None

def get_frame_gray(frame_idx):
    global _cap
    if frame_idx in _frame_cache:
        return _frame_cache[frame_idx]
    if _cap is None:
        _cap = cv2.VideoCapture(VIDEO_PATH)
    _cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = _cap.read()
    if not ret:
        return None
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if len(_frame_cache) > 200:
        _frame_cache.pop(next(iter(_frame_cache)))
    _frame_cache[frame_idx] = gray
    return gray

def build_mask(frame_idx, detections_by_frame, pad=15):
    mask = np.full((IMG_HEIGHT, IMG_WIDTH), 255, dtype=np.uint8)
    for cx, cy in detections_by_frame.get(frame_idx, []):
        x, y = int(cx), int(cy)
        cv2.rectangle(mask, (x-pad, y-pad), (x+pad, y+pad), 0, -1)
    return mask

if len(df):
    detections_by_frame = defaultdict(list)
    for _, r in df.iterrows():
        detections_by_frame[r["frame"]].append((r["cx"], r["cy"]))

    if not os.path.exists(VIDEO_PATH):
        print(f"{VIDEO_PATH} not found")
        video_available = False
    else:
        video_available = True
        print("The video has been found; start creating the mosaic")


The video has been found; start creating the mosaic

In [ ]:
# Paired Homographs + Islands (with Adaptive Rollback)
if len(df) and video_available:
    orb = cv2.ORB_create(nfeatures=2000)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    kp_des_cache = {}

    def get_kp_des(frame_idx):
        if frame_idx in kp_des_cache:
            return kp_des_cache[frame_idx]
        gray = get_frame_gray(frame_idx)
        if gray is None:
            kp_des_cache[frame_idx] = (None, None)
            return None, None
        mask = build_mask(frame_idx, detections_by_frame)
        kp, des = orb.detectAndCompute(gray, mask)
        kp_des_cache[frame_idx] = (kp, des)
        return kp, des

    def estimate_homography(idx_a, idx_b):
        kp_a, des_a = get_kp_des(idx_a)
        kp_b, des_b = get_kp_des(idx_b)
        if des_a is None or des_b is None or len(kp_a) < 8 or len(kp_b) < 8:
            return None, 0, 0
        matches = bf.knnMatch(des_a, des_b, k=2)
        good = [m for m, n in matches if m.distance < 0.75 * n.distance]
        if len(good) < 8:
            return None, len(good), 0
        pts_a = np.float32([kp_a[m.queryIdx].pt for m in good])
        pts_b = np.float32([kp_b[m.trainIdx].pt for m in good])
        H, inlier_mask = cv2.findHomography(pts_a, pts_b, cv2.RANSAC, 4.0)
        n_inliers = int(inlier_mask.sum()) if inlier_mask is not None else 0
        return H, len(good), n_inliers

    detection_frames = sorted(df["frame"].unique())
    lo, hi = detection_frames[0], detection_frames[-1]
    nodes = sorted(set(range(lo, hi + 1, FRAME_STRIDE)) | set(detection_frames))
    print(f"Mosaic nodes: {len(nodes)} (step {FRAME_STRIDE}, plus all frames with detections)")

    island_id_of = {nodes[0]: 0}
    chain_within_island = {nodes[0]: np.eye(3)}
    next_island_id = 1

    for pos in range(1, len(nodes)):
        i = nodes[pos]
        linked = False
        for back in range(1, MAX_LOOKBACK + 1):
            if pos - back < 0:
                break
            j = nodes[pos - back]
            if j not in chain_within_island:
                continue
            H, n_matches, n_inliers = estimate_homography(j, i)
            ratio = n_inliers / n_matches if n_matches else 0
            if H is not None and n_inliers >= MIN_INLIERS and ratio >= MIN_RATIO:
                chain_within_island[i] = chain_within_island[j] @ np.linalg.inv(H)
                island_id_of[i] = island_id_of[j]
                linked = True
                break
        if not linked:
            island_id_of[i] = next_island_id
            chain_within_island[i] = np.eye(3)
            next_island_id += 1

    islands = defaultdict(list)
    for f, isl in island_id_of.items():
        islands[isl].append(f)
    sizes = sorted([len(v) for v in islands.values()], reverse=True)
    print(f"Islands: {len(islands)}, sizes (top 10): {sizes[:10]}")


Missing output!

In [ ]:
# Clarification regarding the islands: similarity-fit of the mosaic to Pythagoras
def fit_similarity(src_xy, dst_xy):
    N = len(src_xy)
    A = np.zeros((2*N, 4)); b = np.zeros(2*N)
    for k in range(N):
        x, y = src_xy[k]; X, Y = dst_xy[k]
        A[2*k] = [x, -y, 1, 0]; A[2*k+1] = [y, x, 0, 1]
        b[2*k], b[2*k+1] = X, Y
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    return sol

def apply_similarity(sol, xy):
    a, bb, tx, ty = sol
    x, y = xy[:, 0], xy[:, 1]
    return np.stack([a*x - bb*y + tx, bb*x + a*y + ty], axis=-1)

if len(df) and video_available:
    def project_point(H, x, y):
        p = H @ np.array([x, y, 1.0]); return p[0]/p[2], p[1]/p[2]

    df["island"] = df["frame"].map(island_id_of)
    mx_my = df.apply(lambda r: project_point(chain_within_island[r["frame"]], r["cx"], r["cy"]), axis=1)
    df["mosaic_x"] = mx_my.apply(lambda t: t[0])
    df["mosaic_y"] = mx_my.apply(lambda t: t[1])

    df["refined_east"] = df["pytha_east"]
    df["refined_north"] = df["pytha_north"]
    df["island_residual_m"] = np.nan

    print(f"{"island":>8} {"n":>4} {"frames":>7} {"residual,m":>11}  status")
    for isl, frames_in_isl in islands.items():
        boxes = df[df["island"] == isl]
        if len(boxes) < 3:
            continue
        src = boxes[["mosaic_x", "mosaic_y"]].values
        dst = boxes[["pytha_east", "pytha_north"]].values
        sol = fit_similarity(src, dst)
        pred = apply_similarity(sol, src)
        resid = np.hypot(pred[:,0]-dst[:,0], pred[:,1]-dst[:,1])
        rms = np.sqrt((resid**2).mean())
        df.loc[boxes.index, "refined_east"] = pred[:,0]
        df.loc[boxes.index, "refined_north"] = pred[:,1]
        df.loc[boxes.index, "island_residual_m"] = rms
        status = "high residual" if rms > 8 else "ok"
        print(f"{isl:>8} {len(boxes):>4} {len(frames_in_isl):>7} {rms:>11.2f}  {status}")

Missing output!

In [ ]:
if len(df):
    cols = ["track_id", "frame", "time_s", "island", "island_residual_m",
            "pytha_east", "pytha_north", "refined_east", "refined_north"]
    df[cols].to_csv("chicken_positions_hybrid.csv", index=False)
    print(f"{len(df)} positions have been saved to chicken_positions_hybrid.csv")

    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    for ax, (ecol, ncol), title in zip(
        axes, [("pytha_east","pytha_north"), ("refined_east","refined_north")],
        ["Only Pythagoras", "Hybrid"]
    ):
        east_t, north_t = latlng_to_local_m(telemetry["lat"].values, telemetry["lng"].values, REF_LAT, REF_LNG)
        ax.plot(east_t, north_t, "-", color="lightgray", linewidth=1, zorder=1)
        for tid, grp in df.groupby("track_id"):
            ax.plot(grp[ecol], grp[ncol], "o-", markersize=3, linewidth=1, alpha=0.7, zorder=2)
        ax.set_title(title); ax.set_xlabel("East, m"); ax.set_ylabel("North, m")
        ax.set_aspect("equal"); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("hybrid_comparison.png", dpi=110)
    plt.show()

Missing output!